In [15]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [16]:
big = pd.read_csv("../week4/women-clothing-accessories.3-class.balanced.csv", sep="\t")
big["sentiment"] = big["sentiment"].replace("neautral", "neutral")

In [17]:
from sklearn.model_selection import train_test_split

texts_train, texts_test, y_train, y_test = train_test_split(
    big["review"], big["sentiment"], test_size=0.2, random_state=42
)

In [4]:
print(len(texts_train), len(texts_test))

72000 18000


In [5]:
vec2 = CountVectorizer(ngram_range=(1, 2))
vec2.fit(texts_train)
X_train = vec2.transform(texts_train)
X_test = vec2.transform(texts_test)
X_train.shape, X_test.shape




((72000, 510954), (18000, 510954))

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

model2 = LogisticRegression(max_iter=1000)
model2.fit(X_train, y_train)
y_pred2 = model2.predict(X_test)
accuracy_score(y_test, y_pred2)

0.7434444444444445

In [7]:
vec3 = CountVectorizer(ngram_range=(1, 2), min_df=3)
X_train3 = vec3.fit_transform(texts_train)
X_test3 = vec3.transform(texts_test)

model3 = LogisticRegression(max_iter=1000)
model3.fit(X_train3, y_train)
y_pred3 = model3.predict(X_test3)

print("Размер словаря:", X_train3.shape[1])
accuracy_score(y_test, y_pred3)

Размер словаря: 77708


0.7321666666666666

In [8]:
vec4 = TfidfVectorizer(ngram_range=(1, 2))
X_train4 = vec4.fit_transform(texts_train)
X_test4 = vec4.transform(texts_test)


model4 = LogisticRegression(max_iter=1000)
model4.fit(X_train4, y_train)
y_pred4 = model4.predict(X_test4)
print("Размер словаря:", X_train4.shape[1])
accuracy_score(y_test, y_pred4)

Размер словаря: 510954


0.753

In [9]:
print(X_train[0].data[:5])
print(X_train4[0].data[:5])

[1 1 2 1 1]
[0.07053487 0.08164137 0.11358985 0.17314919 0.11108409]


In [18]:
import pymorphy3

morph = pymorphy3.MorphAnalyzer()
for word in ["платья", "платьем", "скучная", "скучно", "маломерит"]:
    print(word, "→", morph.parse(word)[0].normal_form)

платья → платье
платьем → платье
скучная → скучный
скучно → скучно
маломерит → маломерить


In [19]:
import re
def lemmatize_text(text):
    words = re.findall(r"\w+", text.lower())
    lemmatized_words = [morph.parse(word)[0].normal_form for word in words]
    return " ".join(lemmatized_words)

lemmatize_text("Платья пришли быстро, но скучная расцветка")

'платье прислать быстро но скучный расцветка'

In [20]:
lem_train = texts_train.apply(lemmatize_text)
lem_test = texts_test.apply(lemmatize_text)

In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

vec5 = TfidfVectorizer(ngram_range=(1, 2))
X_train5 = vec5.fit_transform(lem_train)
X_test5 = vec5.transform(lem_test)

model5 = LogisticRegression(max_iter=1000)
model5.fit(X_train5, y_train)
y_pred5 = model5.predict(X_test5)
accuracy_score(y_test, y_pred5)

0.7525

In [25]:
my_reviews = [
    "Не понравилось, ткань тонкая.",
    "Не плохо: сидит хорошо, ткань приятная.",
    "После первой стирки платье село на два размера. Просто восторг!",
]
expected = ["negative", "positive", "negative"]

review_results = pd.DataFrame({"review": my_reviews, "expected": expected})
review_results["predicted"] = model4.predict(vec4.transform(my_reviews))
review_results["correct"] = review_results["expected"] == review_results["predicted"]

print("'не понравилось' is a learned bigram:", "не понравилось" in vec4.vocabulary_)
review_results

'не понравилось' is a learned bigram: True


,review,expected,predicted,correct
0,"Не понравилось, ткань тонкая.",negative,neutral,False
1,"Не плохо: сидит хорошо, ткань приятная.",positive,positive,True
2,После первой стирки платье село на два размера...,negative,neutral,False
